## Extraction of Data from ERA5 Dataset (GEE) ##

Bands to query from ERA5:
1) temperature_2m (air temperature)
2) skin_temperature (surface temperature)
3) soil_temperature_level_1 (soil temp)
4) volumetric_soil_water_layer_1
5) total_evaporation_sum
6) total_precipitation_sum

In [1]:
# Import Google Earth Engine API and Initialize it. 
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="ey-data-and-ai-challenge")

In [2]:
# Read coordinates and date from water quality training dataset, drop given features.

wq_df = pd.read_csv('../data/water_quality_training_dataset.csv')
wq_df = wq_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
wq_df['id'] = wq_df.index
wq_df.head()

,Latitude,Longitude,Sample Date,id
0,-28.760833,17.730278,02-01-2011,0
1,-26.861111,28.884722,03-01-2011,1
2,-26.450000,28.085833,03-01-2011,2
3,-27.671111,27.236944,03-01-2011,3
4,-27.356667,27.286389,03-01-2011,4


In [12]:
wq_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Latitude     9319 non-null   float64
 1   Longitude    9319 non-null   float64
 2   Sample Date  9319 non-null   object 
 3   id           9319 non-null   int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 291.3+ KB


In [15]:
## need to convert Sample date dd-mm-yyyy to yyyy-mm-dd

wq_df['Sample Date'] = pd.to_datetime(wq_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
wq_df['Sample Date']

0       2011-01-02
1       2011-01-03
2       2011-01-03
3       2011-01-03
4       2011-01-03
           ...    
9314    2015-12-23
9315    2015-12-23
9316    2015-12-23
9317    2015-12-23
9318    2015-12-31
Name: Sample Date, Length: 9319, dtype: object

In [25]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features = []

for index, row in wq_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(10000), #add a 10km buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=1)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=1)).strftime('%Y-%m-%d')
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)         # create feature collection with features



In [26]:
bands = ['temperature_2m',
         'skin_temperature', 
         'soil_temperature_level_1',
         'volumetric_soil_water_layer_1',
         'total_evaporation_sum', 
         'total_precipitation_sum']


era5_collection = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(bands)

In [27]:
def extract_median_values(feat):
    collection = era5_collection.filterDate(feat.get('start_date'), feat.get('end_date'))
    img = collection.reduce(ee.Reducer.median()) # reduce image collection into a single image

    era5_col = img.reduceRegions(collection=ee.FeatureCollection([feat]), reducer=ee.Reducer.first(), scale = 11132)
    
    return era5_col.first()

In [28]:
fc_mapped = fc.map(extract_median_values)

In [29]:
# Process data and export to Google Drive

task = ee.batch.Export.table.toDrive(
    collection=fc_mapped,
    description="era5_csv_export",
    fileNamePrefix= "era5_features_training",
    fileFormat='CSV'
)
task.start()

Same process for extracting data for validation set.